In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras import layers, Model

In [ ]:
df = pd.read_csv("/content/RASFF_window.csv")
display(df.head(10))

,reference,category,type,subject,date,notifying_country,classification,risk_decision,distribution,forAttention,forFollowUp,operator,origin,hazards
0,2025.10497,fats and oils,food,Presence of CBD 30% in oil,31-12-2025 16:05:18,France,information notification for follow-up,potential risk,France,Spain,"France,Spain","France,Spain",Spain,NaN
1,2025.10328,meat and meat products (other than poultry),food,Incorrect use-by date on the label of Hespenwo...,31-12-2025 16:04:21,Belgium,non-compliance notification,potential risk,"Belgium,France,Luxembourg,Netherlands",NaN,"France,Luxembourg,Netherlands",Belgium,Belgium,NaN
2,2025.10496,cereals and bakery products,food,Wrong expiry date on hazelnut tart from Belgium,31-12-2025 16:00:00,Belgium,information notification for follow-up,potential risk,Luxembourg,NaN,NaN,Belgium,Belgium,NaN
3,2025.10495,"nuts, nut products and seeds",food,Aflatoxin in Groundnut Runner Raw Split from A...,31-12-2025 15:48:09,Netherlands,information notification for attention,serious,NaN,NaN,NaN,"Argentina,Netherlands",Argentina,"Aflatoxin B1 - {mycotoxins},aflatoxin total ..."
4,2025.10494,"nuts, nut products and seeds",food,Aflatossine B1 e totali in pistacchi senza gus...,31-12-2025 14:58:24,Italy,border rejection notification,serious,NaN,Iran,NaN,"Iran,Italy",Iran,"Aflatoxin B1 - {mycotoxins},aflatoxin total ..."
5,2025.10493,fruits and vegetables,food,Total aflatoxins in dried figs from Türkiye,31-12-2025 14:28:05,Italy,border rejection notification,serious,NaN,NaN,NaN,Türkiye,Türkiye,aflatoxin total - {mycotoxins}
6,2025.10492,fruits and vegetables,food,Pesticide residues (chlorpyrifos) in egg plant...,31-12-2025 14:11:29,Belgium,border rejection notification,potentially serious,NaN,"Burkina Faso,France",NaN,"Belgium,Burkina Faso,France",Burkina Faso,chlorpyrifos unauthorised substance - {pestic...
7,2025.10489,fruits and vegetables,food,Presenza di ocratossina in fichi secchi dalla ...,31-12-2025 13:47:30,Italy,border rejection notification,serious,NaN,Türkiye,NaN,"Italy,Türkiye",Türkiye,ochratoxin A - {mycotoxins}
8,2025.10488,herbs and spices,food,Alérgenos no declarados (avellana y almendra) ...,31-12-2025 12:13:14,Spain,information notification for attention,serious,"Netherlands,Spain","France,Spain","France,Netherlands","France,Madagascar,Mauritius,Spain",Spain,"almond traces - {allergens},hazelnut traces ..."
9,2025.10487,fruits and vegetables,food,Oxamyl in strawberries from Egypt.,31-12-2025 11:23:46,Finland,information notification for attention,serious,"Belgium,Finland,France,Germany,Netherlands","Belgium,Finland,France,Germany,Netherlands",Belgium,"Belgium,Egypt,Finland,Netherlands",Egypt,oxamyl unauthorised substance - {pesticide re...


In [ ]:
df.count()

,0
reference,5328
category,5328
type,5328
subject,5328
date,5328
notifying_country,5328
classification,5328
risk_decision,5328
distribution,3681
forAttention,3836


## Limpieza de datos

In [ ]:
# Encoding (Convertir palabras a números)
le = LabelEncoder()

# Limpieza de "hazards"
def clean_hazard(text):
    if pd.isna(text): return "Unknown"
    if "{" in str(text):
        # Extrae lo que está entre llaves { }
        return str(text).split('{')[-1].split('}')[0]
    return "Other"


# Simplificar el riesgo (risk_decision)
df['risk_level'] = df['risk_decision'].fillna('not serious')

# Eliminar columnas con nulos
df_clean = df.drop(columns=['forAttention', 'forFollowUp', 'operator', 'distribution'])

# Limpiar espacios en blanco invisibles en todas las columnas de texto
columnas_texto = ['category', 'type', 'notifying_country', 'origin', 'hazards', 'risk_decision']
for col in columnas_texto:
    df[col] = df[col].astype(str).str.strip()

In [ ]:
encoders = {}
features_to_encode = ['category', 'type', 'notifying_country', 'origin', 'hazards']

for col in features_to_encode:
    encoders[col] = LabelEncoder()
    df[col] = encoders[col].fit_transform(df[col].astype(str))

# Encoder especial para la respuesta final (y)
le_risk = LabelEncoder()
y = le_risk.fit_transform(df['risk_decision'].astype(str))
X = df[features_to_encode]

# Entrenar el modelo
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler

features_list = ['category', 'type', 'notifying_country', 'origin', 'hazards']
X_meta_raw = df[features_list].values

scaler = StandardScaler()
X_meta_scaled = scaler.fit_transform(X_meta_raw)

In [ ]:
# Preparación de datos (Texto + Variables numéricas)
max_words = 1000
max_len = 20

# Tokenizar el texto (Subject)
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['subject'].astype(str))
X_text = tf.keras.preprocessing.sequence.pad_sequences(tokenizer.texts_to_sequences(df['subject'].astype(str)), maxlen=max_len)

## Transformer


In [ ]:
# Arquitectura con Atención
input_text = layers.Input(shape=(max_len,), name="Texto")
input_meta = layers.Input(shape=(5,), name="Variables")

# Rama de texto con Embedding
embedding = layers.Embedding(max_words, 64)(input_text)

# Capa de Atención (Multi-Head)
attention = layers.MultiHeadAttention(num_heads=2, key_dim=64)(embedding, embedding) #Auto-atención
attention = layers.GlobalAveragePooling1D()(attention)

# Unimos el resultado de la atención con las 5 variables tabulares
concat = layers.Concatenate()([attention, input_meta])

# Capa de decisión final
dense = layers.Dense(32, activation='relu')(concat)
outputs = layers.Dense(len(le_risk.classes_), activation='softmax')(dense)

# Crear y compilar
model = Model(inputs=[input_text, input_meta], outputs=outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Entrenar
model.fit([X_text, X_meta_scaled], y, epochs=20, validation_split=0.2, batch_size=32)

Epoch 1/20
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.4510 - loss: 1.2995 - val_accuracy: 0.5216 - val_loss: 1.1938
Epoch 2/20
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.5509 - loss: 1.0880 - val_accuracy: 0.5769 - val_loss: 1.0692
Epoch 3/20
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.6229 - loss: 0.9339 - val_accuracy: 0.5826 - val_loss: 1.0527
Epoch 4/20
134/134 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.6661 - loss: 0.8500 - val_accuracy: 0.5910 - val_loss: 1.0994
Epoch 5/20
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.6863 - loss: 0.7986 - val_accuracy: 0.5872 - val_loss: 1.1510
Epoch 6/20
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7034 - loss: 0.7630 - val_accuracy: 0.5901 - val_loss: 1.1876
Epoch 7/20
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7191 - loss: 0.7285 - val_accuracy: 0.5675 - val_loss: 1.2630
Epoch 8/20
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7276 - loss: 0.6982 - val_accu

In [ ]:
import numpy as np

In [ ]:
def predecir_riesgo_con_texto(subject_text, category, prod_type, notifying_country, origin, hazard_type):
    # 1. Procesar texto del Subject
    seq = tokenizer.texts_to_sequences([subject_text])
    text_input = tf.keras.preprocessing.sequence.pad_sequences(seq, maxlen=max_len)

    try:
        # 2. Transformar variables SIGUIENDO EL ORDEN:
        # ['category', 'type', 'notifying_country', 'origin', 'hazards']
        meta_data = [
            encoders['category'].transform([str(category)])[0],
            encoders['type'].transform([str(prod_type)])[0],
            encoders['notifying_country'].transform([str(notifying_country)])[0],
            encoders['origin'].transform([str(origin)])[0],
            encoders['hazards'].transform([str(hazard_type)])[0]
        ]

        # 3. Escalar y Predecir
        meta_input = scaler.transform(np.array([meta_data]))
        res = model.predict([text_input, meta_input], verbose=0)

        return le_risk.inverse_transform([np.argmax(res)])[0]

    except ValueError as e:
        return f"Error de etiqueta: {e}. Revisa que los nombres existan en el dataset."

In [ ]:
# EJEMPLO 1
print(f"--- RESULTADO DE LA PREDICCIÓN ---")
print(predecir_riesgo_con_texto(
    subject_text = "Pesticide residues in vegetables",
    category = "fruits and vegetables",
    prod_type = "food",
    notifying_country = "France",
    origin = "India",
    hazard_type = "Aflatoxin   - {mycotoxins}"
))

--- RESULTADO DE LA PREDICCIÓN ---
serious


In [ ]:
# EJEMPLO 2
print(f"--- RESULTADO DE LA PREDICCIÓN ---")
print(predecir_riesgo_con_texto(
    subject_text = "Formaldehido en vajilla de China",
    category = "food contact materials",
    prod_type = "food contact material",
    notifying_country = "Spain",
    origin = "China",
    hazard_type = "formaldehyde  increasing migration - {migration}"
))

--- RESULTADO DE LA PREDICCIÓN ---
potential risk


In [ ]:
print("Categorías disponibles en tu modelo:")
print(list(encoders['hazards'].classes_)[:20])

Categorías disponibles en tu modelo:
['10-hydroxy-hexacannabinol (10-OH-HHC) unauthorised novel food ingredient - {novel food}', '10-hydroxy-hexacannabinol (10-OH-HHC) unauthorised novel food ingredient - {novel food},cannabidiol (CBD)  unauthorised novel food ingredient - {novel food}', '10-hydroxy-hexacannabinol (10-OH-HHC) unauthorised novel food ingredient - {novel food},cannabidiol (CBD)  unauthorised novel food ingredient - {novel food},muscimol unauthorised substance - {composition}', '10-hydroxy-hexacannabinol (10-OH-HHC) unauthorised novel food ingredient - {novel food},tetrahydrocanabinol (THC)   - {biological contaminants}', '2,4,6-trichloroanisole (TCA)  - {chemical contamination (other)},procymidone   - {pesticide residues}', '2-chloroethanol   - {pesticide residues},ethylene oxide   - {pesticide residues}', '3-monochlor-1,2-propanediol (3-MCPD)   - {industrial contaminants}', '3-monochlor-1,2-propanediol (3-MCPD)   - {industrial contaminants},colour E 110 - Sunset Yellow 

In [ ]:
# Busca cualquier peligro que contenga la palabra 'acephate'
palabra_a_buscar = 'formaldehyde'
coincidencias = [h for h in encoders['hazards'].classes_ if palabra_a_buscar.lower() in h.lower()]

print(f"Peligros encontrados en el encoder:")
print(coincidencias)

Peligros encontrados en el encoder:
['formaldehyde  increasing migration - {migration}', 'formaldehyde  increasing migration - {migration},melamine  increasing migration - {migration}', 'formaldehyde  migration - {migration}', 'formaldehyde  migration - {migration},melamine  migration - {migration}']
